# Modelo 1 — Detecção de Anomalias em Contratos

Treina e avalia o `IsolationForest` de `models/anomaly_detection.py` sobre a Gold real (`iceberg.gold.fato_contrato` + dimensões, via Trino). Não supervisionado: não existe rótulo de "contrato irregular" em nenhuma fonte — o modelo aprende só a partir da distribuição dos próprios dados.

Pré-requisito: stack do lakehouse no ar (Hive Metastore + Trino, Gold já construída pelo `dbt build`). Ver `.env.example` para `TRINO_HOST`/`TRINO_PORT`.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
load_dotenv(dotenv_path=project_root / ".env")

from models import anomaly_detection as ad

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


## 1. Extração das features (Gold + Silver via Trino)

`fato_contrato` já traz `valor_contrato`/`flag_emergency`; `dim_credor` traz o histórico de infração (SCD2); `dim_modalidade`, a modalidade. `tipo_objeto` e as datas de vigência ainda não estão modeladas na Gold, então a query junta direto com `iceberg.silver.contratos` para pegá-los — ver `anomaly_detection.FEATURE_QUERY`.


In [ ]:
raw = ad.extract_features()
print(f"Contratos com valor_contrato preenchido: {len(raw)}")
raw.head()


In [ ]:
print("Nulos por coluna (%):")
print((raw.isna().mean() * 100).round(1))
print()
print("% flag_emergency=True:", round(100 * raw['flag_emergency'].fillna(False).mean(), 2))
print("% historico_credor_infringement=True:", round(100 * raw['historico_credor_infringement'].fillna(False).mean(), 2))


In [ ]:
raw['modalidade'].value_counts().head(10)


In [ ]:
raw['tipo_objeto'].value_counts().head(10)


## 2. Engenharia de features

`build_feature_matrix` converte o bruto em matriz numérica: `valor_contrato`, `dias_vigencia` (término − início), one-hot de `modalidade`/`tipo_objeto` (categorias raras agrupadas em `OUTROS`) e as duas flags booleanas.


In [ ]:
X = ad.build_feature_matrix(raw)
print(f"Shape da matriz de features: {X.shape}")
X.describe().T


## 3. Treino — Isolation Forest

`contamination='auto'` (padrão do scikit-learn) — não fixamos a priori qual fração dos contratos é anômala, como recomenda a dica 7.3 do enunciado (não há rótulo ground-truth para calibrar isso).


In [ ]:
model = ad.train_model(X, contamination="auto")
scores = ad.score_anomalia(model, X)

resultado = raw.copy()
resultado["score_anomalia"] = scores.values
resultado[["id_contrato_origem", "ano", "valor_contrato", "modalidade", "score_anomalia"]].head()


## 4. Avaliação

Sem rótulo de verdade, a avaliação é por distribuição do score e checagem de sanidade: os contratos mais anômalos fazem sentido de negócio (valor muito fora da curva, vigência atípica, credor com histórico de infração, contratação emergencial)?


In [ ]:
print("Distribuição do score de anomalia:")
print(resultado["score_anomalia"].describe())


In [ ]:
faixas = pd.cut(resultado["score_anomalia"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
faixas.value_counts().sort_index()


In [ ]:
colunas_contexto = [
    "id_contrato_origem", "ano", "valor_contrato", "modalidade", "tipo_objeto",
    "flag_emergency", "historico_credor_infringement", "score_anomalia",
]
resultado.sort_values("score_anomalia", ascending=False)[colunas_contexto].head(20)


### Checagem de sanidade — o score reage a sinais de negócio conhecidos?

Contratos emergenciais e credores com histórico de infração deveriam, em média, sair com score mais alto que a base geral — não é garantido (o modelo não sabe o que essas flags significam, só que são incomuns), mas é um sinal de que o modelo captura algo relevante e não só ruído.


In [ ]:
comparativo = pd.DataFrame({
    "score_medio_geral": [resultado["score_anomalia"].mean()],
    "score_medio_emergencial": [resultado.loc[resultado["flag_emergency"] == True, "score_anomalia"].mean()],
    "score_medio_credor_com_infracao": [resultado.loc[resultado["historico_credor_infringement"] == True, "score_anomalia"].mean()],
})
comparativo.round(4)


## 5. Persistência do modelo

Salva o modelo treinado e a lista de colunas de feature (necessária para re-aplicar o mesmo one-hot em score futuro) em `models/artifacts/` — git-ignorado, reproduzível a partir deste notebook ou de `python -m models.anomaly_detection`.


In [ ]:
ad.save_model(model, list(X.columns))
print(f"Modelo salvo em: {ad.ARTIFACT_PATH}")


## 6. Gravação em produção (fora deste notebook)

Este notebook é só para treino/avaliação exploratórios — não escreve na Gold. Em produção, `models.anomaly_detection.run()` já chama `write_scores()` internamente e grava `(id_contrato_origem, ano, score_anomalia)` em `iceberg.gold.score_anomalia_contrato`; `fato_contrato.sql` (dbt) faz `LEFT JOIN` nessa tabela, então o score aparece na próxima vez que a Gold for reconstruída. Isso roda automaticamente pela DAG `dags/dag_ml_inference.py` (task `score_anomalias`), disparada por Dataset logo após a DAG 3 (Gold) terminar — ver a task `refresh_fato_contrato` na mesma DAG.


## Achados e limitações

- **Sem rótulo ground-truth** — a avaliação acima é indireta (distribuição + checagem de sanidade), não uma métrica de classificação (precision/recall). O enunciado já antecipa isso: recomenda validar os contratos de maior score com analistas de controle antes de confiar no ranking.
- **`tipo_objeto` truncado no top-8** — categorias raras viram `OUTROS`; se o negócio achar isso grosseiro, o corte é ajustável em `anomaly_detection.TOP_TIPO_OBJETO`.
- **Próximos passos** (fora do escopo deste notebook): DAG de inferência no Airflow (tarefa 23), gravar `score_anomalia` de volta na `fato_contrato` (tarefa 24), e tratar desbalanceamento se o modelo for convertido para supervisionado no futuro (tarefa 28).
